## Otimização
<p>Este notebook tem como objetivo exemplificar e comparar implementações convencionais e otimizadas já demonstradas no projeto.</p>
<p>Serão comparadas diferentes etratégias e respectivos tempos de execução para duas operações.</p>
<ol>
<li></li>
</ol>
<p>A análise será realizada comparativamente com a estratégia <code><b>Broadcast Join</b></code>, comparando tempos de execução</p>

In [0]:
# Imports
from time import perf_counter
from pyspark.sql import functions as F

In [0]:
# Definição de constantes
catalog      = "ANP_Combustiveis"
silverSchema = "02_silver"

priceTable     = (f"`{catalog}`.`{silverSchema}`.`precos_revenda`")
municipalTalbe = (f"`{catalog}`.`{silverSchema}`.`municipios`")

timeResults = []

##### Data reading

In [0]:
# Carregamento de dados de precos
pricesDf = (
    spark.table(priceTable)
    .select(
        "codigo_ibge",
        "familia_combustivel",
        "data_coleta",
        "valor_venda",
    )
)

# Carregamento de dados de municipios
municipalDf = (
    spark.table(municipalTalbe)
    .select(
        "codigo_ibge",
        "nome_regiao",
    )
)

In [0]:
pricesDf.agg(
    F.max("codigo_ibge"),
    F.max("familia_combustivel"),
    F.max("data_coleta"),
    F.max("valor_venda"),
).collect()

municipalDf.agg(
    F.max("codigo_ibge"),
    F.max("nome_regiao"),
).collect()

### Comparação 1 - Função Python vs. função nativa spark
<p>Na camada 'Gold', realizamos a extração do ano da data de coleta. Iremos comparar a operação com a implementação python com aquela realizada pelo Spark, que tende a ser otimizada pelo Catalyst. 

In [0]:
def getYear(data):
    if data is None:
        return None

    return data.year


getYear_python = F.udf(getYear, "int")
resumeDf_python = (
    pricesDf
    .withColumn("ano_referencia", getYear_python("data_coleta"))
    .groupBy("ano_referencia", "familia_combustivel")
    .agg(
        F.avg("valor_venda").alias("preco_medio"),
        F.count("*").alias("quantidade_observacoes")
    )
)
#resumeDf_python.explain(mode="formatted")

start = perf_counter()
pythonAgg = (
    resumeDf_python
    .agg(
        F.count("*").alias("total_grupos"),
        F.sum("preco_medio").alias("soma_precos"),
        F.sum("quantidade_observacoes").alias("total_observacoes")
    )
    .collect()[0]
)
timePython = perf_counter() - start

resumeSparkDf = (
    pricesDf
    .withColumn("ano_referencia", F.year("data_coleta"))
    .groupBy("ano_referencia", "familia_combustivel")
    .agg(
        F.avg("valor_venda").alias("preco_medio"),
        F.count("*").alias("quantidade_observacoes")
    )
)
#resumoNativoDf.explain(mode="formatted")

start = perf_counter()
sparkAgg = (
    resumeSparkDf
    .agg(
        F.count("*").alias("total_grupos"),
        F.sum("preco_medio").alias("soma_precos"),
        F.sum("quantidade_observacoes").alias("total_observacoes"),
    )
    .collect()[0]
)
timeSpark = perf_counter() - start

print('Comparação dos tempos de execução:')
print(f"\t-Python: {timePython:.3f} s")
print(f"\t-Spark: {timeSpark:.3f} s")

### Comparação 2 - Join
<p>É de conhecimento geral que o join é um dos gargalos de performance do Spark. Uma das estrategias para contornar isto é a realização do broadcast join.</p>
<p>Aqui, comparamos uma estrategia de join convencional com uma estrategia de broadcast join.</p>

In [0]:
# Join convencional
normalJoinDf = (
    pricesDf.hint("merge").alias("p")
    .join(
        municipalDf.hint("merge").alias("m"),
        on="codigo_ibge",
        how="inner",
    )
    .groupBy(
        "nome_regiao",
        "familia_combustivel",
    )
    .agg(
        F.avg("valor_venda").alias("preco_medio"),
        F.count("*").alias("quantidade_observacoes"),
    )
)

start = perf_counter()
normalJoinAgg = (
    normalJoinDf
    .agg(
        F.count("*").alias("total_grupos"),
        F.sum("preco_medio").alias("soma_precos"),
        F.sum("quantidade_observacoes").alias("total_observacoes"),
    )
    .collect()[0]
)
normalJoinTime = perf_counter() - start


# Join otimizado (Broadcast)
optimizedJoinDf = (
    pricesDf.alias("p")
    .join(
        F.broadcast(
            municipalDf
        ).alias("m"),
        on="codigo_ibge",
        how="inner",
    )
    .groupBy(
        "nome_regiao",
        "familia_combustivel",
    )
    .agg(
        F.avg("valor_venda").alias("preco_medio"),
        F.count("*").alias("quantidade_observacoes"),
    )
)

start = perf_counter()
optimizedJoinAgg = (
    optimizedJoinDf
    .agg(
        F.count("*").alias("total_grupos"),
        F.sum("preco_medio").alias("soma_precos"),
        F.sum("quantidade_observacoes").alias("total_observacoes"),
    )
    .collect()[0]
)
optimizedJoinTime = perf_counter() - start

print('Comparação dos tempos de execução:')
print(f"\t-Join ''Convencional': {normalJoinTime:.3f} s")
print(f"\t-Broadcast join: {optimizedJoinTime:.3f} s")